In [13]:
import os
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mne.time_frequency import psd_array_welch

# --- Paths ---
input_file = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1.edf"
output_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1_processed"

# Create output directory if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# --- Frequency bands ---
freq_bands = {
    'Delta': (1, 4),
    'Theta': (4, 8),
    'Alpha': (8, 13),
    'Beta': (13, 30),
    'Gamma': (30, 40)
}

# --- Channel Mapping ---
channel_mapping = {
    'Pz': 'Pz-REF', 'P3': 'P3-REF', 'P4': 'P4-REF', 'O1': 'O1-REF', 'Oz': 'Oz-REF', 'O2': 'O2-REF',
    'PO3': 'Po3-REF', 'PO4': 'Po4-REF', 'PO7': 'Po7-REF', 'PO8': 'Po8-REF',
    'Fp1': 'Fp1-REF', 'Fp2': 'Fp2-REF', 'Fz': 'Fz-REF', 'F3': 'F3-REF', 'F4': 'F4-REF',
    'AF3': 'Af3-REF', 'AF4': 'Af4-REF',
    'T7': 'T7-REF', 'T8': 'T8-REF', 'TP7': 'Tp7-REF', 'TP8': 'Tp8-REF',
    'C3': 'C3-REF', 'C4': 'C4-REF', 'CP3': 'Cp3-REF', 'CP4': 'Cp4-REF',
    'EOG_left': 'EOG_left-REF', 'EOG_right': 'EOG_right-REF', 'EMG_lower_right': 'EMG_lower_right-REF',
    'EMG_lower_left': 'EMG_lower_left-REF', 'EMG_ref': 'EMG_ref-REF', 'AFz': 'Fpz-REF'
}

# --- Functions ---
def rename_channels(raw, mapping):
    # Get the current channel names
    current_channels = raw.ch_names
    
    # Create a list for the new channel names
    new_channels = []
    
    # Loop through the current channel names and apply the mapping
    for ch in current_channels:
        # If a channel exists in the mapping, rename it, else keep the original name
        new_channels.append(mapping.get(ch, ch))
    
    # Apply the new channel names to the raw data
    raw.rename_channels(dict(zip(current_channels, new_channels)))
    
    return raw

def preprocess_eeg(file_path):
    raw = mne.io.read_raw_edf(file_path, preload=True)
    
    # Rename channels based on the provided mapping
    raw = rename_channels(raw, channel_mapping)
    
    # Select the relevant channels manually (based on new names)
    selected_channels = [
        "Pz-REF", "P3-REF", "P4-REF", "O1-REF", "Oz-REF", "O2-REF",
        "Po3-REF", "Po4-REF", "Po7-REF", "Po8-REF",
        "Fpz-REF", "Fp1-REF", "Fp2-REF", "Fz-REF", "F3-REF", "F4-REF",
        "Af3-REF", "Af4-REF",
        "T7-REF", "T8-REF", "Tp7-REF", "Tp8-REF",
        "C3-REF", "C4-REF", "Cp3-REF", "Cp4-REF"
    ]
    
    raw.pick_channels(selected_channels)
    raw.set_eeg_reference(ref_channels='average')
    raw.filter(l_freq=1, h_freq=45, fir_design='firwin')
    raw.resample(250, npad="auto")
    
    return raw

def calculate_flattened_band_power(raw):
    psds, freqs = psd_array_welch(raw.get_data(), sfreq=raw.info['sfreq'], fmin=1, fmax=50, n_fft=2048)
    
    psds_normalized = (psds - np.mean(psds, axis=1, keepdims=True)) / np.std(psds, axis=1, keepdims=True)
    
    flattened_band_powers = []
    for band_name, band_range in freq_bands.items():
        band_psd = psds_normalized[:, (freqs >= band_range[0]) & (freqs <= band_range[1])]
        flattened_band_powers.extend(band_psd.mean(axis=1))
    
    return np.array(flattened_band_powers)

# --- Main Execution ---

print(f"Processing file: {input_file}")

# Preprocess
raw = preprocess_eeg(input_file)

# Save preprocessed EEG
preprocessed_output_file = os.path.join(output_folder, "morningnap_part1_processed.fif")
raw.save(preprocessed_output_file, overwrite=True)

# Extract flattened EEG features
flattened_band_powers = calculate_flattened_band_power(raw)

# Save features
eeg_features_df = pd.DataFrame([flattened_band_powers])
eeg_features_df['filename'] = os.path.basename(input_file)

features_output_file = os.path.join(output_folder, 'eeg_features_DONDERS_TEST.csv')
eeg_features_df.to_csv(features_output_file, index=False)

print(f"Saved processed EEG file to {preprocessed_output_file}")
print(f"Saved EEG features to {features_output_file}")


Processing file: /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1.edf
Extracting EDF parameters from /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2717366  =      0.000 ...  5434.732 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- 

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.7s


Writing /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1_processed/morningnap_part1_processed.fif


/var/folders/2h/5twk6xg13mxcmtl80ctk427h0000gn/T/ipykernel_10464/3360195774.py:98: RuntimeWarning: This filename (/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1_processed/morningnap_part1_processed.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  raw.save(preprocessed_output_file, overwrite=True)


Closing /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1_processed/morningnap_part1_processed.fif
[done]
Effective window size : 8.192 (s)
Saved processed EEG file to /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1_processed/morningnap_part1_processed.fif
Saved EEG features to /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1_processed/eeg_features_DONDERS_TEST.csv


In [7]:
raw = mne.io.read_raw_edf(input_file, preload=True)

Extracting EDF parameters from /Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Dream Database from Donders/Data/PSG/s_02/c_02/morningnap_part1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 2717366  =      0.000 ...  5434.732 secs...


In [23]:
# Load the CSV file into a DataFrame
zhang_wamsley_df = pd.read_csv('/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/eeg_features_reordered.csv')

# Filter the DataFrame for the specific filename
filtered_df1 = zhang_wamsley_df[zhang_wamsley_df['filename'] == 'subject249_SO3']

# Display the filtered DataFrame
print(filtered_df1)

           0         1         2         3         4         5         6  \
114  2.63047  2.735169  2.421578  2.674216  2.739189  2.359624  2.736124   

            7         8         9  ...       121       122       123  \
114  2.359471  2.672749  2.049254  ... -0.438236 -0.443032 -0.452179   

          124       125       126       127       128       129  \
114 -0.413512 -0.460281 -0.445335 -0.435393 -0.439419 -0.443796   

           filename  
114  subject249_SO3  

[1 rows x 131 columns]


In [24]:
filtered_df1.to_csv('eeg_zw_test1.csv', index=False)

In [19]:
zhang_wamsley_df.head()

data = zhang_wamsley_df[zhang_wamsley_df['filename'] == 'subject249_SO3']
data

,0,1,2,3,4,5,6,7,8,9,...,121,122,123,124,125,126,127,128,129,filename
114,2.63047,2.735169,2.421578,2.674216,2.739189,2.359624,2.736124,2.359471,2.672749,2.049254,...,-0.438236,-0.443032,-0.452179,-0.413512,-0.460281,-0.445335,-0.435393,-0.439419,-0.443796,subject249_SO3


In [18]:
print(zhang_wamsley_df['filename'].unique())  # Check unique filenames


['subject010_Morning' 'subject010_NREM' 'subject010_REM' 'subject010_SO10'
 'subject010_SO2' 'subject010_SO3' 'subject010_SO4' 'subject010_SO5'
 'subject010_SO6' 'subject010_SO7' 'subject010_SO8' 'subject010_SO9'
 'subject026_Morning' 'subject026_NREM' 'subject026_SO1' 'subject026_SO10'
 'subject026_SO2' 'subject026_SO3' 'subject026_SO4' 'subject026_SO5'
 'subject026_SO6' 'subject026_SO8' 'subject026_SO9' 'subject030_SO2'
 'subject030_SO3' 'subject030_SO5' 'subject030_SO8' 'subject042_Morning'
 'subject042_SO1' 'subject042_SO2' 'subject042_SO3' 'subject042_SO4'
 'subject083_REM' 'subject083_SO2' 'subject088_Morning' 'subject088_REM'
 'subject088_SO1' 'subject088_SO10' 'subject088_SO2' 'subject088_SO3'
 'subject088_SO4' 'subject088_SO5' 'subject088_SO6' 'subject088_SO7'
 'subject088_SO8' 'subject088_SO9' 'subject107_Morning' 'subject107_NREM'
 'subject107_REM' 'subject107_SO1' 'subject107_SO10' 'subject107_SO2'
 'subject107_SO3' 'subject107_SO4' 'subject107_SO6' 'subject107_SO7'
 'subje